## Decision Trajectories & Parameter Sweeps in 3D Behavioral Space

### 1. Constructing the 3D Behavioral Projection Space
To inspect how the detector's operational state evolves under changing filtering gates, we adapted the concept of continuous trajectory curves. Instead of tracking steps over chronological time, we use systematic threshold sweeps ($T_{\text{Conf}}$ and $T_{\text{IoU}}$) as a pseudo-temporal parameter.

At every threshold step, we capture the behavior of each class through a 6-dimensional state vector:
$$\mathbf{s} = \big( \text{Recall}, \, \text{Precision}, \, \text{F}_1\text{-Score}, \, \overline{\text{Conf}}, \, \overline{\text{IoU}}, \, \text{Error Share} \big)$$

We also inject a theoretical ideal reference point—**★ Perfect Classification** ($100\%$ across all performance metrics, $0\%$ error share). Standardizing these states across all sweep steps and projecting them via **3D PCA** reveals the underlying dynamics:
- **PC1:** Represents overall detection capability and precision, serving as the main distance axis toward the theoretical benchmark.
- **PC2:** Reflects localization quality, tracking the trade-off between strict overlap criteria and candidate retention.
- **PC3:** Captures the internal balance between confidence calibration and false-positive suppression, untangling trajectories that overlapped in 2D.
- **Cumulative Explained Variance:** Moving to three principal components preserves over $90\%$ of the total behavioral variance, giving an uncompressed view of how each category responds to filtering.

*(Note: In the confidence sweep, extreme outliers such as `diningtable` and `boat` were excluded from the PCA fit to avoid skewing the embedding axes and compressing the remaining class paths.)*

---

### 2. Intuitive Interpretation: What Do These Sweeps Actually Tell Us?

The two plots illustrate a fundamental difference between what the model predicts and how strictly we evaluate its bounding boxes: **Tuning confidence actively changes the model's operating point, whereas demanding higher IoU acts merely as a localization stress test.**

#### Confidence Sweep ($T_{\text{Conf}}: 0.25 \to 0.90$ at fixed $\text{IoU} = 0.50$) — *The Active Tuning Lever*
* **Why it actually improves classification quality:** Raising the confidence threshold actively filters out the detector's guesses. At low thresholds ($0.25$), the model outputs plenty of noisy, half-baked candidate boxes (many False Positives). As we push $T_{\text{Conf}}$ higher, precision rises sharply: the detector keeps only the detections it is genuinely certain about, causing trajectories to climb upward toward the benchmark star.
* **The Sweet Spot vs. Over-Filtering:**
  * **Vehicles, Animals & Person (`car`, `aeroplane`, `dog`):** These classes benefit the most from tightening confidence. They reach a clear optimal operating window (around $0.60$–$0.80$) where precision is high without sacrificing too many real objects.
  * **Indoor Objects (`chair`, `sofa`):** For ambiguous indoor furniture, the model is rarely confident to begin with. Raising the threshold does not turn poor detections into good ones; instead, the model immediately drops them into *Missed Targets*, peeling off into low-recall territory before ever achieving high precision.

#### IoU Sweep ($T_{\text{IoU}}: 0.20 \to 0.90$ at fixed $\text{Conf} = 0.25$) — *The Passive Localization Stress Test*
* **Why raising IoU does not "improve" the model:** The IoU threshold is an evaluation criterion, not a model setting. Increasing the required IoU does not make YOLO's predicted bounding boxes any tighter—it only punishes the boxes YOLO already predicted.
* **The Inevitable Downward Drop:**
  * Demanding $\text{IoU} \ge 0.70$ does not increase precision; it simply turns correct classifications with slightly loose bounding boxes into failures (*Missed (IoU < T)*). Every single trajectory therefore collapses downward away from the benchmark star as the threshold tightens.
* **Rigid Boxes vs. Deformable Contours:**
  * **Vehicles (`bus`, `train`):** Rectangular, rigid objects naturally align well with axis-aligned bounding boxes. Their trajectories remain stable across moderate IoU thresholds before dropping.
  * **Organic / Deformable Shapes (`person`, `cat`):** Poses change constantly and limbs extend outward, making pixel-perfect box overlap rare. Enforcing strict IoU requirements breaks these detections almost immediately, demonstrating that the failure mode here is geometric alignment rather than semantic misidentification.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =============================================================================
# 1. DATENBASIS LADEN & GRUNDSTRUKTUREN
# =============================================================================
BASE_DIR = Path("__file__").resolve().parent if "__file__" in globals() else Path.cwd()
CSV_PATH = BASE_DIR / "voc_behavioral_results.csv"

if not CSV_PATH.exists():
    alt_path = Path(r"C:\Users\Simon Mader\Documents\AI\3. Semester\AI_and_visualization\AI_visualization\Andres-Simon\phase1\voc_behavioral_results.csv")
    if alt_path.exists():
        CSV_PATH = alt_path

df = pd.read_csv(CSV_PATH)

VOC_CLASSES_SEMANTIC = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car",
    "cat", "chair", "cow", "diningtable", "dog", "horse", "motorbike",
    "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]
VOC_CLASSES_ALPHABETICAL = sorted(VOC_CLASSES_SEMANTIC)
BACKGROUND_LABEL = "None (Background)"

SUPER_CATEGORY_MAP = {
    "aeroplane": "Vehicles", "bicycle": "Vehicles", "boat": "Vehicles",
    "bus": "Vehicles", "car": "Vehicles", "motorbike": "Vehicles", "train": "Vehicles",
    "bird": "Animals & Person", "cat": "Animals & Person", "cow": "Animals & Person",
    "dog": "Animals & Person", "horse": "Animals & Person", "person": "Animals & Person", "sheep": "Animals & Person",
    "bottle": "Indoor Objects", "chair": "Indoor Objects", "diningtable": "Indoor Objects",
    "pottedplant": "Indoor Objects", "sofa": "Indoor Objects", "tvmonitor": "Indoor Objects",
}

SUPER_COLORS = {
    "Animals & Person": "#3b82f6",  # Kräftiges Blau
    "Vehicles": "#22c55e",          # Kräftiges Grün
    "Indoor Objects": "#f59e0b",    # Kräftiges Bernstein / Orange
}

df["Predicted_Class"] = df["Predicted_Class"].fillna("None").astype(str)
df["Ground_Truth"] = df["Ground_Truth"].fillna(BACKGROUND_LABEL).astype(str)
df["Confidence"] = pd.to_numeric(df["Confidence"], errors="coerce")
df["IoU"] = pd.to_numeric(df["IoU"], errors="coerce").fillna(0.0).clip(0.0, 1.0)
df["Candidate_Confidence"] = df["Confidence"]

blind_mask = (df["Outcome"] == "Missed Target") & (df["IoU"] <= 0.0)
df.loc[blind_mask, "Candidate_Confidence"] = np.nan

if "Is_Duplicate_Candidate_Row" not in df.columns:
    df["Is_Duplicate_Candidate_Row"] = False
else:
    df["Is_Duplicate_Candidate_Row"] = df["Is_Duplicate_Candidate_Row"].fillna(False).astype(bool)

base_for_sweep = df[~df["Is_Duplicate_Candidate_Row"]].copy()

# =============================================================================
# 2. DYNAMISCHE EVALUIERUNG
# =============================================================================
def evaluate_dynamic_state(d_in: pd.DataFrame, conf_t: float, iou_t: float) -> pd.DataFrame:
    d = d_in.copy()
    conf = pd.to_numeric(d["Candidate_Confidence"], errors="coerce").to_numpy(dtype=float)
    iou = pd.to_numeric(d["IoU"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    orig = d["Outcome"].astype(str).to_numpy()
    gt_cls = d["Ground_Truth"].astype(str).to_numpy()
    cand_cls = d["Candidate_Class"].fillna("None").astype(str).to_numpy() if "Candidate_Class" in d.columns else d["Predicted_Class"].astype(str).to_numpy()
    pred_cls = d["Predicted_Class"].fillna("None").astype(str).to_numpy()
    eff_cls = np.where(cand_cls != "None", cand_cls, pred_cls)

    is_gt = (gt_cls != BACKGROUND_LABEL) & (orig != "False Positive")
    is_fp = ~is_gt
    has_cand = np.isfinite(conf) & (conf >= 0.0)
    pass_conf = has_cand & (conf >= conf_t)
    pass_iou = iou >= iou_t
    has_cls = eff_cls != "None"
    corr_cls = eff_cls == gt_cls

    dyn_out = np.full(len(d), "Blind Spot (IoU=0)", dtype=object)
    dyn_out[is_gt & (~has_cand | (iou <= 0.0))] = "Blind Spot (IoU=0)"
    dyn_out[is_gt & has_cand & (iou > 0.0) & (~pass_conf)] = "Missed (Conf<T)"
    
    act_gt = is_gt & pass_conf
    dyn_out[act_gt & pass_iou & has_cls & corr_cls] = "TP (Conf, IoU>T)"
    dyn_out[act_gt & pass_iou & has_cls & (~corr_cls)] = "Misclassification"
    dyn_out[act_gt & (~(pass_iou & has_cls))] = "Missed (IoU<T)"

    dyn_out[is_fp & pass_conf] = "FP (Hallucination)"
    dyn_out[is_fp & (~pass_conf)] = "Filtered FP"

    d["Dynamic_Outcome"] = dyn_out
    d["Effective_Pred_Class"] = eff_cls
    return d

# =============================================================================
# 3. SWEEP & 3D PCA (MIT AUSSCHLUSSLISTE)
# =============================================================================
def generate_projected_paths_3d(base_df, sequence_type="conf", exclude_classes=None):
    if exclude_classes is None:
        exclude_classes = []

    active_classes = [c for c in VOC_CLASSES_ALPHABETICAL if c not in exclude_classes]

    if sequence_type == "conf":
        seq_vals = np.round(np.arange(0.25, 0.901, 0.05), 2)
        pairs = [(float(v), 0.50) for v in seq_vals]
        labels = [f"Conf={v:.2f}" for v in seq_vals]
    else:
        seq_vals = np.round(np.arange(0.20, 0.901, 0.05), 2)
        pairs = [(0.25, float(v)) for v in seq_vals]
        labels = [f"IoU={v:.2f}" for v in seq_vals]

    path_records = []
    for step_idx, ((c_t, i_t), step_lbl) in enumerate(zip(pairs, labels)):
        step_df = evaluate_dynamic_state(base_df, c_t, i_t)
        gt_df = step_df[step_df["Ground_Truth"] != BACKGROUND_LABEL]

        for cls_name in active_classes:
            cls_gt = gt_df[gt_df["Ground_Truth"] == cls_name]
            gt_count = len(cls_gt)
            tps = cls_gt[(cls_gt["Dynamic_Outcome"] == "TP (Conf, IoU>T)") & (cls_gt["Effective_Pred_Class"] == cls_name)]
            tp_count = len(tps)

            pred_as_cls = step_df[(step_df["Effective_Pred_Class"] == cls_name) & step_df["Candidate_Confidence"].notna() & (step_df["Candidate_Confidence"] >= c_t)]
            fp_count = max(0, len(pred_as_cls) - tp_count)

            recall = tp_count / gt_count * 100.0 if gt_count else 0.0
            precision = tp_count / (tp_count + fp_count) * 100.0 if (tp_count + fp_count) > 0 else 0.0
            f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            mean_conf = float(tps["Candidate_Confidence"].mean() * 100.0) if tp_count else 0.0
            mean_iou = float(tps["IoU"].mean() * 100.0) if tp_count else 0.0
            err_share = fp_count / (tp_count + fp_count) * 100.0 if (tp_count + fp_count) > 0 else 0.0

            path_records.append({
                "Class": cls_name,
                "Super_Category": SUPER_CATEGORY_MAP[cls_name],
                "Step_Idx": step_idx,
                "Step_Label": step_lbl,
                "Recall": round(recall, 2),
                "Precision": round(precision, 2),
                "F1": round(f1, 2),
                "Mean_Conf": mean_conf,
                "Mean_IoU": mean_iou,
                "Error_Share": err_share,
            })

    path_df = pd.DataFrame(path_records)
    feat_cols = ["Recall", "Precision", "F1", "Mean_Conf", "Mean_IoU", "Error_Share"]
    ideal_vec = np.array([[100.0, 100.0, 100.0, 100.0, 100.0, 0.0]])

    scaler = StandardScaler()
    std_states = scaler.fit_transform(np.vstack([path_df[feat_cols].to_numpy(dtype=float), ideal_vec]))
    
    pca = PCA(n_components=3, random_state=42)
    coords = pca.fit_transform(std_states)

    if coords[-1, 0] < 0:
        coords[:, 0] *= -1.0

    path_df["PC1"] = coords[:-1, 0]
    path_df["PC2"] = coords[:-1, 1]
    path_df["PC3"] = coords[:-1, 2]
    star_xyz = (coords[-1, 0], coords[-1, 1], coords[-1, 2])

    return path_df, star_xyz, pca.explained_variance_ratio_, active_classes

# Daten berechnen: Im conf-Fall ohne "diningtable" und ohne "boat"
df_conf_3d, star_conf_3d, var_conf_3d, classes_conf = generate_projected_paths_3d(
    base_for_sweep, "conf", exclude_classes=["diningtable", "boat"]
)
df_iou_3d, star_iou_3d, var_iou_3d, classes_iou = generate_projected_paths_3d(
    base_for_sweep, "iou", exclude_classes=[]
)

# =============================================================================
# 4. INTERAKTIVER 3D-DUAL-PANEL PLOT
# =============================================================================
fig_3d = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=(
        f"Confidence Threshold Progression (3D, exkl. diningtable & boat)<br><sup>T_conf: 0.25 → 0.90 | {(var_conf_3d.sum())*100:.1f}% Variance Explained</sup>",
        f"IoU Requirement Progression (3D)<br><sup>T_IoU: 0.20 → 0.90 | {(var_iou_3d.sum())*100:.1f}% Variance Explained</sup>"
    ),
    horizontal_spacing=0.04
)

def add_paths_to_3d_scene(fig_obj, path_df, star_xyz, active_classes, col_idx):
    for cls_name in active_classes:
        class_path = path_df[path_df["Class"] == cls_name].sort_values("Step_Idx")
        super_cat = SUPER_CATEGORY_MAP[cls_name]
        base_color = SUPER_COLORS[super_cat]

        customdata = np.stack([
            class_path["Class"],
            class_path["Step_Label"],
            class_path["Precision"],
            class_path["Recall"],
            class_path["F1"],
            class_path["Mean_IoU"],
        ], axis=-1)

        # 3D Pfadlinie mit Datenpunkten
        fig_obj.add_trace(
            go.Scatter3d(
                x=class_path["PC1"],
                y=class_path["PC2"],
                z=class_path["PC3"],
                mode="lines+markers",
                name=f"{cls_name} ({super_cat})",
                legendgroup=super_cat,
                showlegend=False,
                line=dict(color=base_color, width=4.5),
                marker=dict(
                    size=np.linspace(3.5, 7.5, len(class_path)),
                    color=class_path["F1"],
                    colorscale="Viridis",
                    cmin=0, cmax=100,
                    showscale=False,
                    line=dict(width=0.8, color="#ffffff")
                ),
                customdata=customdata,
                hovertemplate=(
                    "Class: <b>%{customdata[0]}</b><br>"
                    "Step: <b>%{customdata[1]}</b><br>"
                    "Precision: %{customdata[2]:.1f}% | Recall: %{customdata[3]:.1f}%<br>"
                    "F1: <b>%{customdata[4]:.1f}%</b> | Mean IoU: %{customdata[5]:.1f}%"
                    "<extra></extra>"
                )
            ),
            row=1, col=col_idx
        )

        # Text-Label am Endpunkt (strengster Schwellenwert)
        fig_obj.add_trace(
            go.Scatter3d(
                x=[float(class_path["PC1"].iloc[-1])],
                y=[float(class_path["PC2"].iloc[-1])],
                z=[float(class_path["PC3"].iloc[-1])],
                mode="text",
                text=[f"<b>{cls_name}</b>"],
                textposition="top center",
                textfont=dict(size=12, color="#f8fafc"),
                showlegend=False,
                hoverinfo="none"
            ),
            row=1, col=col_idx
        )

    # Benchmark: ★ Perfect Classification
    fig_obj.add_trace(
        go.Scatter3d(
            x=[star_xyz[0]],
            y=[star_xyz[1]],
            z=[star_xyz[2]],
            mode="markers+text",
            name="★ Perfect Classification",
            text=["<b>★ Perfect Classification</b>"],
            textposition="top center",
            textfont=dict(size=12, color="#facc15"),
            marker=dict(
                symbol="diamond",
                size=12,
                color="#facc15",
                line=dict(width=2, color="#ffffff")
            ),
            showlegend=False,
            hoverinfo="text",
            hovertext="Theoretical Perfect Benchmark:<br>100% Recall, 100% Precision, 0% Error"
        ),
        row=1, col=col_idx
    )

# Subplots befüllen
add_paths_to_3d_scene(fig_3d, df_conf_3d, star_conf_3d, classes_conf, col_idx=1)
add_paths_to_3d_scene(fig_3d, df_iou_3d, star_iou_3d, classes_iou, col_idx=2)

# Dummy-Spuren für Legende
for cat_name, cat_col in SUPER_COLORS.items():
    fig_3d.add_trace(
        go.Scatter3d(
            x=[None], y=[None], z=[None], mode="lines+markers",
            name=cat_name,
            line=dict(color=cat_col, width=4),
            marker=dict(size=5, color=cat_col)
        )
    )
fig_3d.add_trace(
    go.Scatter3d(
        x=[None], y=[None], z=[None], mode="markers",
        name="★ Perfect Benchmark",
        marker=dict(symbol="diamond", size=8, color="#facc15")
    )
)

# Dark-Theme Konfiguration
scene_layout_1 = dict(
    xaxis=dict(title=f"PC1 ({var_conf_3d[0]*100:.1f}%)", backgroundcolor="#161b22", gridcolor="#30363d", showbackground=True),
    yaxis=dict(title=f"PC2 ({var_conf_3d[1]*100:.1f}%)", backgroundcolor="#161b22", gridcolor="#30363d", showbackground=True),
    zaxis=dict(title=f"PC3 ({var_conf_3d[2]*100:.1f}%)", backgroundcolor="#161b22", gridcolor="#30363d", showbackground=True),
)

scene_layout_2 = dict(
    xaxis=dict(title=f"PC1 ({var_iou_3d[0]*100:.1f}%)", backgroundcolor="#161b22", gridcolor="#30363d", showbackground=True),
    yaxis=dict(title=f"PC2 ({var_iou_3d[1]*100:.1f}%)", backgroundcolor="#161b22", gridcolor="#30363d", showbackground=True),
    zaxis=dict(title=f"PC3 ({var_iou_3d[2]*100:.1f}%)", backgroundcolor="#161b22", gridcolor="#30363d", showbackground=True),
)

fig_3d.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0e1117",
    height=800,
    margin=dict(t=90, b=40, l=20, r=20),
    scene=scene_layout_1,
    scene2=scene_layout_2,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.06,
        xanchor="center",
        x=0.5,
        font=dict(size=11, color="#f0f6fc"),
        bgcolor="rgba(31, 36, 45, 0.9)",
        bordercolor="#30363d",
        borderwidth=1
    )
)

fig_3d.show()

## The same in 2D

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =============================================================================
# 1. DATENBASIS LADEN & GRUNDSTRUKTUREN
# =============================================================================
BASE_DIR = Path("__file__").resolve().parent if "__file__" in globals() else Path.cwd()
CSV_PATH = BASE_DIR / "voc_behavioral_results.csv"

if not CSV_PATH.exists():
    alt_path = Path(r"C:\Users\Simon Mader\Documents\AI\3. Semester\AI_and_visualization\AI_visualization\Andres-Simon\phase1\voc_behavioral_results.csv")
    if alt_path.exists():
        CSV_PATH = alt_path

df = pd.read_csv(CSV_PATH)

VOC_CLASSES_SEMANTIC = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car",
    "cat", "chair", "cow", "diningtable", "dog", "horse", "motorbike",
    "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]
VOC_CLASSES_ALPHABETICAL = sorted(VOC_CLASSES_SEMANTIC)
BACKGROUND_LABEL = "None (Background)"

SUPER_CATEGORY_MAP = {
    "aeroplane": "Vehicles", "bicycle": "Vehicles", "boat": "Vehicles",
    "bus": "Vehicles", "car": "Vehicles", "motorbike": "Vehicles", "train": "Vehicles",
    "bird": "Animals & Person", "cat": "Animals & Person", "cow": "Animals & Person",
    "dog": "Animals & Person", "horse": "Animals & Person", "person": "Animals & Person", "sheep": "Animals & Person",
    "bottle": "Indoor Objects", "chair": "Indoor Objects", "diningtable": "Indoor Objects",
    "pottedplant": "Indoor Objects", "sofa": "Indoor Objects", "tvmonitor": "Indoor Objects",
}

SUPER_COLORS = {
    "Animals & Person": "#3b82f6",  # Kräftiges Blau
    "Vehicles": "#22c55e",          # Kräftiges Grün
    "Indoor Objects": "#f59e0b",    # Kräftiges Bernstein / Orange
}

df["Predicted_Class"] = df["Predicted_Class"].fillna("None").astype(str)
df["Ground_Truth"] = df["Ground_Truth"].fillna(BACKGROUND_LABEL).astype(str)
df["Confidence"] = pd.to_numeric(df["Confidence"], errors="coerce")
df["IoU"] = pd.to_numeric(df["IoU"], errors="coerce").fillna(0.0).clip(0.0, 1.0)
df["Candidate_Confidence"] = df["Confidence"]

blind_mask = (df["Outcome"] == "Missed Target") & (df["IoU"] <= 0.0)
df.loc[blind_mask, "Candidate_Confidence"] = np.nan

if "Is_Duplicate_Candidate_Row" not in df.columns:
    df["Is_Duplicate_Candidate_Row"] = False
else:
    df["Is_Duplicate_Candidate_Row"] = df["Is_Duplicate_Candidate_Row"].fillna(False).astype(bool)

base_for_sweep = df[~df["Is_Duplicate_Candidate_Row"]].copy()

# =============================================================================
# 2. DYNAMISCHE EVALUIERUNG
# =============================================================================
def evaluate_dynamic_state(d_in: pd.DataFrame, conf_t: float, iou_t: float) -> pd.DataFrame:
    d = d_in.copy()
    conf = pd.to_numeric(d["Candidate_Confidence"], errors="coerce").to_numpy(dtype=float)
    iou = pd.to_numeric(d["IoU"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    orig = d["Outcome"].astype(str).to_numpy()
    gt_cls = d["Ground_Truth"].astype(str).to_numpy()
    cand_cls = d["Candidate_Class"].fillna("None").astype(str).to_numpy() if "Candidate_Class" in d.columns else d["Predicted_Class"].astype(str).to_numpy()
    pred_cls = d["Predicted_Class"].fillna("None").astype(str).to_numpy()
    eff_cls = np.where(cand_cls != "None", cand_cls, pred_cls)

    is_gt = (gt_cls != BACKGROUND_LABEL) & (orig != "False Positive")
    is_fp = ~is_gt
    has_cand = np.isfinite(conf) & (conf >= 0.0)
    pass_conf = has_cand & (conf >= conf_t)
    pass_iou = iou >= iou_t
    has_cls = eff_cls != "None"
    corr_cls = eff_cls == gt_cls

    dyn_out = np.full(len(d), "Blind Spot (IoU=0)", dtype=object)
    dyn_out[is_gt & (~has_cand | (iou <= 0.0))] = "Blind Spot (IoU=0)"
    dyn_out[is_gt & has_cand & (iou > 0.0) & (~pass_conf)] = "Missed (Conf<T)"
    
    act_gt = is_gt & pass_conf
    dyn_out[act_gt & pass_iou & has_cls & corr_cls] = "TP (Conf, IoU>T)"
    dyn_out[act_gt & pass_iou & has_cls & (~corr_cls)] = "Misclassification"
    dyn_out[act_gt & (~(pass_iou & has_cls))] = "Missed (IoU<T)"

    dyn_out[is_fp & pass_conf] = "FP (Hallucination)"
    dyn_out[is_fp & (~pass_conf)] = "Filtered FP"

    d["Dynamic_Outcome"] = dyn_out
    d["Effective_Pred_Class"] = eff_cls
    return d

# =============================================================================
# 3. SWEEP & STANDARDIZED PCA
# =============================================================================
def generate_projected_paths(base_df, sequence_type="conf"):
    if sequence_type == "conf":
        seq_vals = np.round(np.arange(0.25, 0.901, 0.05), 2)
        pairs = [(float(v), 0.50) for v in seq_vals]
        labels = [f"Conf={v:.2f}" for v in seq_vals]
    else:
        seq_vals = np.round(np.arange(0.20, 0.901, 0.05), 2)
        pairs = [(0.25, float(v)) for v in seq_vals]
        labels = [f"IoU={v:.2f}" for v in seq_vals]

    path_records = []
    for step_idx, ((c_t, i_t), step_lbl) in enumerate(zip(pairs, labels)):
        step_df = evaluate_dynamic_state(base_df, c_t, i_t)
        gt_df = step_df[step_df["Ground_Truth"] != BACKGROUND_LABEL]

        for cls_name in VOC_CLASSES_ALPHABETICAL:
            cls_gt = gt_df[gt_df["Ground_Truth"] == cls_name]
            gt_count = len(cls_gt)
            tps = cls_gt[(cls_gt["Dynamic_Outcome"] == "TP (Conf, IoU>T)") & (cls_gt["Effective_Pred_Class"] == cls_name)]
            tp_count = len(tps)

            pred_as_cls = step_df[(step_df["Effective_Pred_Class"] == cls_name) & step_df["Candidate_Confidence"].notna() & (step_df["Candidate_Confidence"] >= c_t)]
            fp_count = max(0, len(pred_as_cls) - tp_count)

            recall = tp_count / gt_count * 100.0 if gt_count else 0.0
            precision = tp_count / (tp_count + fp_count) * 100.0 if (tp_count + fp_count) > 0 else 0.0
            f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            mean_conf = float(tps["Candidate_Confidence"].mean() * 100.0) if tp_count else 0.0
            mean_iou = float(tps["IoU"].mean() * 100.0) if tp_count else 0.0
            err_share = fp_count / (tp_count + fp_count) * 100.0 if (tp_count + fp_count) > 0 else 0.0

            path_records.append({
                "Class": cls_name,
                "Super_Category": SUPER_CATEGORY_MAP[cls_name],
                "Step_Idx": step_idx,
                "Step_Label": step_lbl,
                "Recall": round(recall, 2),
                "Precision": round(precision, 2),
                "F1": round(f1, 2),
                "Mean_Conf": round(mean_conf, 2),
                "Mean_IoU": round(mean_iou, 2),
                "Error_Share": round(err_share, 2),
            })

    path_df = pd.DataFrame(path_records)
    feat_cols = ["Recall", "Precision", "F1", "Mean_Conf", "Mean_IoU", "Error_Share"]
    ideal_vec = np.array([[100.0, 100.0, 100.0, 100.0, 100.0, 0.0]])

    scaler = StandardScaler()
    std_states = scaler.fit_transform(np.vstack([path_df[feat_cols].to_numpy(dtype=float), ideal_vec]))
    
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(std_states)

    if coords[-1, 0] < 0:
        coords[:, 0] *= -1.0

    path_df["PC1"] = coords[:-1, 0]
    path_df["PC2"] = coords[:-1, 1]
    star_xy = (coords[-1, 0], coords[-1, 1])

    return path_df, star_xy, pca.explained_variance_ratio_

# Pfaddaten berechnen
df_conf, star_conf, var_conf = generate_projected_paths(base_for_sweep, "conf")
df_iou, star_iou, var_iou = generate_projected_paths(base_for_sweep, "iou")

# =============================================================================
# 4. INTERAKTIVER DUAL-PANEL PLOT IN ECHTEM STREAMLIT-STIL
# =============================================================================
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Confidence Threshold Progression<br><sup>(T_conf: 0.25 → 0.90 at fixed IoU = 0.50 — {(var_conf[0]+var_conf[1])*100:.1f}% Variance Explained)</sup>",
        f"IoU Requirement Progression<br><sup>(T_IoU: 0.20 → 0.90 at fixed Conf = 0.25 — {(var_iou[0]+var_iou[1])*100:.1f}% Variance Explained)</sup>"
    ),
    horizontal_spacing=0.08
)

def add_paths_to_subplot(fig_obj, path_df, star_xy, col_idx, show_leg_group=False):
    for cls_name in VOC_CLASSES_ALPHABETICAL:
        class_path = path_df[path_df["Class"] == cls_name].sort_values("Step_Idx")
        super_cat = SUPER_CATEGORY_MAP[cls_name]
        base_color = SUPER_COLORS[super_cat]

        customdata = np.stack([
            class_path["Class"],
            class_path["Step_Label"],
            class_path["Precision"],
            class_path["Recall"],
            class_path["F1"],
            class_path["Mean_IoU"],
        ], axis=-1)

        # 1. Glatte Spline-Kurve mit sichtbaren Punkten
        fig_obj.add_trace(
            go.Scatter(
                x=class_path["PC1"],
                y=class_path["PC2"],
                mode="lines+markers",
                name=f"{cls_name} ({super_cat})",
                legendgroup=super_cat,
                showlegend=False,
                line=dict(
                    color=base_color,
                    width=2.4,
                    shape="spline",
                    smoothing=0.85
                ),
                marker=dict(
                    size=np.linspace(5.0, 11.5, len(class_path)),
                    color=class_path["F1"],
                    colorscale="Viridis",
                    cmin=0,
                    cmax=100,
                    showscale=False,
                    line=dict(width=0.9, color="#ffffff")
                ),
                customdata=customdata,
                hovertemplate=(
                    "Class: <b>%{customdata[0]}</b><br>"
                    "Step: <b>%{customdata[1]}</b><br>"
                    "Precision: %{customdata[2]:.1f}% | Recall: %{customdata[3]:.1f}%<br>"
                    "F1: <b>%{customdata[4]:.1f}%</b> | Mean IoU: %{customdata[5]:.1f}%"
                    "<extra></extra>"
                )
            ),
            row=1, col=col_idx
        )

        # 2. Beschriftung am Endpunkt
        fig_obj.add_annotation(
            x=float(class_path["PC1"].iloc[-1]),
            y=float(class_path["PC2"].iloc[-1]),
            text=f"<b>{cls_name}</b>",
            showarrow=False,
            yshift=10,
            font=dict(size=12, color="#f8fafc"),
            bgcolor="rgba(15, 23, 42, 0.75)",
            bordercolor="rgba(148, 163, 184, 0.3)",
            borderwidth=0.8,
            row=1, col=col_idx
        )

    # 3. Perfekter Stern Benchmark
    fig_obj.add_trace(
        go.Scatter(
            x=[star_xy[0]],
            y=[star_xy[1]],
            mode="markers+text",
            name="★ Perfect Classification",
            text=["<b>★ Perfect Classification</b>"],
            textposition="bottom left",
            textfont=dict(size=12, color="#facc15"),
            marker=dict(
                symbol="star",
                size=24,
                color="#facc15",
                line=dict(width=1.8, color="#ffffff")
            ),
            showlegend=False,
            hoverinfo="text",
            hovertext="Theoretical Perfect Boundary:<br>100% Recall, 100% Precision, 0% Error"
        ),
        row=1, col=col_idx
    )

# Beide Panels befüllen
add_paths_to_subplot(fig, df_conf, star_conf, col_idx=1)
add_paths_to_subplot(fig, df_iou, star_iou, col_idx=2)

# Dummy-Spuren für eine aufgeräumte Superkategorie-Legende
for cat_name, cat_col in SUPER_COLORS.items():
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None], mode="lines+markers",
            name=cat_name,
            line=dict(color=cat_col, width=3),
            marker=dict(size=8, color=cat_col, line=dict(width=1, color="white"))
        )
    )
fig.add_trace(
    go.Scatter(
        x=[None], y=[None], mode="markers",
        name="★ Perfect Benchmark",
        marker=dict(symbol="star", size=14, color="#facc15", line=dict(width=1, color="white"))
    )
)

# Dark-Theme Layout
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0e1117",
    plot_bgcolor="#161b22",
    height=750,
    margin=dict(t=80, b=70, l=60, r=40),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.12,
        xanchor="center",
        x=0.5,
        font=dict(size=13, color="#f0f6fc"),
        bgcolor="rgba(31, 36, 45, 0.9)",
        bordercolor="#30363d",
        borderwidth=1
    )
)

fig.update_xaxes(
    title_text=f"Behavioral Dimension 1 ({var_conf[0]*100:.1f}% var)",
    gridcolor="#30363d", zerolinecolor="#475569", row=1, col=1
)
fig.update_yaxes(
    title_text=f"Behavioral Dimension 2 ({var_conf[1]*100:.1f}% var)",
    gridcolor="#30363d", zerolinecolor="#475569", row=1, col=1
)

fig.update_xaxes(
    title_text=f"Behavioral Dimension 1 ({var_iou[0]*100:.1f}% var)",
    gridcolor="#30363d", zerolinecolor="#475569", row=1, col=2
)
fig.update_yaxes(
    title_text=f"Behavioral Dimension 2 ({var_iou[1]*100:.1f}% var)",
    gridcolor="#30363d", zerolinecolor="#475569", row=1, col=2
)

fig.show()